# Lesson 03 - ఏజెంటిక్ డిజైన్ ప్యాటర్న్స్

ఈ పాఠంలో, సమర్ధవంతమైన AI ఏజెంట్లను నిర్మించడానికి మూడు మూలప్రామాణిక డిజైన్ ప్యాటర్న్లను పరిశీలిస్తాము:

1. **స్పష్ట ఏజెంట్ సూచనలు** — ఏజెంట్ ప్రవర్తనను మార్గదర్శనం చేసే ఖచ్చితమైన, పాత్ర నిర్వచించే ప్రాంప్టులను తయారుచేసుకోవడం  
2. **Pydantic మోడల్స్‌తో నిర్మిత అవుట్పుట్** — ఏజెంట్లు ఊహించదగిన, ధృవీకరించిన డేటాను తిరిగి ఇచ్చేటట్లు నిర్ధారించడం  
3. **ఒక్కటి బాధ్యత ఉన్న ఏజెంట్లు** — ఒక్కటే పనిని బాగా చేసే దృష్టితో ఏజెంట్లను డిజైన్ చేయడం  

ప్రతి ప్యాటర్న్ను **ప్రయాణ గమ్యస్థాన సిఫార్సు చేస్తున్న అనువర్తనం** సన్నివేశంలో వరసగా వర్తింప చేయబోతున్నాం, ఇది గమ్యస్థానాలను సూచించడం, అందుబాటు తనిఖీ చేయడం మరియు లాజిస్టిక్స్ నిర్వహించడం చేసే వ్యవస్థను క్రమంగా నిర్మిస్తుంది.


## సెటప్


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## నమూనా 1: స్పష్టమైన ఏజెంట్ సూచనలు

అత్యంత ప్రభావవంతమైన నమూనా కూడా చాలా సులభమైనది: మీ ఏజెంట్ కోసం స్పష్టమైన, వివరమైన సూచనలు రాయడం.

మంచి సూచనలు నిర్వచిస్తాయి:
- **ఎవరు** ఏజెంట్ (వ్యక్తిత్వం మరియు శైలి)
- **ఏమి** చేయాలి (దశల వారీ బాధ్యతలు)
- **ఎలా** ప్రవర్తించాలి (పరిమితులు మరియు శైలి)

క్రింద, ప్రతి స్పందనను రూపొందించే స్పష్టమైన సూచనలతో ఒక ప్రయాణ కంక్యర్జ్ ఏజెంట్ ను సృష్టిస్తున్నాము.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Pydantic మోడల్స్‌తో నిర్మాణాత్మక అవుట్పుట్

ఫ్రీ-ఫారమ్ టెక్స్ట్ సంభాషణకు ఉపయోగకరం, కానీ డౌన్‌స్ట్రీమ్ సిస్టమ్‌లకు నిర్మిత డేటా అవసరం.
**Pydantic మోడల్స్**ను ఒక **టూల్ ఫంక్షన్**తో జతచేయడం ద్వారా, మనం చేయగలము:

- ఏజెంట్ అవుట్పుట్ కోసం ఖచ్చితమైన స్కీమాను నిర్వచించడం
- స్పందనలను స్వయంచాలకంగా నిర్ధారించడం
- ఏజెంట్ ఫలితాలను అప్లికేషన్ లాజిక్‌లో నమ్మకంగా సమగ్రపరచడం

మరియు ఏజెంట్ తన సిఫారసులు నిజమైన డేటాలో స్థాపించుకునేందుకు గమ్యస్థానం వివరాలను అందించే ఒక టూల్‌ను కూడా పరిచయం చేస్తాము.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## నమూనా 3: సింగిల్ రిస్పాన్సిబిలిటీ ఏజెంట్లు

సంక్లిష్ట పనులు పలు ప్రత్యేక ఏజెంట్ల మధ్య పని విభజన ద్వారా లాభదాయకం అవుతాయి, ప్రతి ఒక్కరికి ఒకే బాధ్యత ఉంటుంది:

- ప్రదేశాలు మరియు అందుబాటు గురించి తెలుసుకునే **గమ్యస్థానం నిపుణుడు**
- విమానాలు, హోటల్స్, మరియు ప్రయాణ పట్టికలను నిర్వహించే **లాజిస్టిక్స్ ప్లానర్**

ఇది సాఫ్ట్‌వేర్ ఇంజనీరింగ్ సూత్రమైన *అసూఢితల వేరు చేయడం*ని ప్రతిబింబిస్తుంది — ప్రతి ఏజెంట్‌ను స్వతంత్రంగా పరీక్షించడం, నిర్వహించడం మరియు మెరుగుపరచడం సులభం అవుతుంది.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## సారాంశం

ఈ పాఠంలో మేము ట్రావెల్ సిఫార్సు చేసే సందర్భంలో మూడు ఏజెంటిక్ డిజైన్ పాటర్న్లను ఉపయోగించాము:

| పాటర్న్ | కీలక ఆలోచన | ప్రయోజనం |
|---|---|---|
| **స్పష్టమైన సూచనలు** | వ్యక్తిత్వం, బాధ్యతలు, మరియు పరిమితులను ముందుగా నిర్వచించండి | స్థాయికి అనుగుణమైన, బ్రాండ్‌కు అనుకూలమైన ఏజెంట్ ప్రవర్తన |
| **సంఘటిత అవుట్పుట్** | ప్రతిస్పందన ఫార్మాట్‌గా Pydantic మోడల్స్ ఉపయోగించండి | ధృవీకరించబడిన, యంత్రం చదవగల ఫలితాలు |
| **ఏక బాధ్యతా** | ప్రతి ఏజెంట్‌కు ఒక హెచ్చరికపూరిత పని ఇవ్వండి | పరీక్షించడానికి, నిర్వహించడానికి, మరియు కలిసి రాయడానికి సులభం |

ఈ పాటర్న్లు సహజంగానే కలిపి పనిచేస్తాయి — మీరు స్పష్టమైన సూచనలను సంఘటిత అవుట్పుట్‌తో ఏక బాధ్యతా ఏజెంట్‌లో కలిపి బలమైన, ఉత్పత్తి-సిద్ధ సిస్టమ్స్ రూపొందించవచ్చు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**డిస్క్లైమర్**:
ఈ పత్రాన్ని AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించారు. మనం ఖచ్చితత్వానికి యత్నించినప్పటికీ, ఆటోమేటెడ్ అనువాదాలలో తప్పిదాలు లేదా అసమర్థతలు ఉండగలవని దయచేసి గమనించండి. ఒరిజినల్ పత్రం దాని స్వదేశీ భాషలోనే అధికారం ఉన్న వనరుగా పరిగణించాలి. సమగ్ర సమాచారం కోసం, వృత్తిపరమైన మానవ అనువాదం సిఫార్సు చేయబడుతుంది. ఈ అనువాదం ఉపయోగించడం వల్ల ఏర్పడిన ఏవైనా అపోహలు లేదా తప్పుదోవలను దాఖలుచేసుకోవడం మేము బాధ్యులు కాదు.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
